# 🛡️ CyberSentinel — Fine-Tuning do Qwen3-1.7B (Arquitetura Clássica)

**Objetivo:** Realizar o ajuste fino do modelo `unsloth/Qwen3-1.7B` transformando-o no **CyberSentinel**, um analista SOC sênior virtual especialista em cibersegurança.

**Por que Qwen3-1.7B?**
- ✅ Arquitetura Transformer clássica (sem SSM/Mamba2 ou MTP/NextN)
- ✅ Exportação GGUF estável e compatível com `llama.cpp` e `llama-cpp-python`
- ✅ Excelente custo-benefício de desempenho para execução em CPU local
- ✅ Suporte nativo a pensamento estruturado (Thinking Mode)

**Abordagem:**
- **Modelo Base:** `unsloth/Qwen3-1.7B` (não-instruct, treinamos o comportamento nós mesmos)
- **Otimização:** QLoRA (Rank=16, Alpha=32) via Unsloth
- **Dataset:** `sambanovasystems/attackqa` (Q&A real de MITRE ATT&CK)
- **Split:** 70% Treino / 30% Validação
- **Exportação:** GGUF Q4_K_M → arquivo `cybersentinel_qwen3.Q4_K_M.gguf`

**Resultado:** Baixe o arquivo `.gguf` gerado e renomeie para `qwen_finetuned.gguf` dentro da pasta `backend/models/`.

## 1. 📦 Instalação de Dependências

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Instalação local
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## 2. 🤖 Carregamento do Modelo Base

Usamos o `Qwen3-1.7B` — arquitetura Transformer clássica sem camadas SSM/Mamba ou MTP.
Isso garante que a exportação GGUF será perfeitamente compatível com `llama-cpp-python`.

In [ ]:
from unsloth import FastLanguageModel
import torch

# ─── Configuração do Modelo ───────────────────────────────────────────────────
MAX_SEQ_LENGTH = 4096   # Janela de contexto para logs e relatórios longos
DTYPE = None            # Detecção automática (bfloat16 em T4/A100, float16 em outros)
LOAD_IN_4BIT = True     # QLoRA: 4-bit para economizar VRAM na GPU T4 gratuita

# IMPORTANTE: Usamos 'unsloth/Qwen3-1.7B' (base), não o Instruct.
# O fine-tuning com o dataset especializado do CyberSentinel vai
# moldar o comportamento instruct diretamente, evitando conflito com
# os pesos de instrução genérica do modelo Instruct original.
MODEL_ID = "unsloth/Qwen3-1.7B"

print(f"[CyberSentinel] Carregando modelo base: {MODEL_ID}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DTYPE,
    load_in_4bit = LOAD_IN_4BIT,
)
print(f"[CyberSentinel] Modelo carregado com sucesso!")

## 3. 🔧 Configuração dos Adaptadores LoRA (PEFT / QLoRA)

Aplicamos os adaptadores LoRA apenas nas matrizes de atenção e projeção.
Isso reduz os parâmetros treináveis para ~1-2% do modelo, acelerando o treinamento drasticamente.

In [ ]:
print("[CyberSentinel] Configurando adaptadores QLoRA...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                        # Rank do LoRA — balanço ideal entre qualidade e memória
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",   # Camadas de Atenção
        "gate_proj", "up_proj", "down_proj",       # Camadas Feed-Forward (MLP)
    ],
    lora_alpha = 32,               # Fator de escala (2x o rank = boa prática)
    lora_dropout = 0,              # Dropout desativado — padrão otimizado do Unsloth
    bias = "none",                 # Sem bias treinável
    use_gradient_checkpointing = "unsloth",  # Economiza 30% de VRAM
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)
print("[CyberSentinel] Adaptadores QLoRA configurados com sucesso!")

## 4. 📊 Carregamento e Preparação do Dataset

Usamos o dataset `sambanovasystems/attackqa` que contém Q&A reais baseados no framework MITRE ATT&CK.
Convertemos cada entrada para o formato de chat do Qwen3, injetando o **System Prompt do CyberSentinel**.

In [ ]:
# ─── System Prompt do CyberSentinel ──────────────────────────────────────────
# Este prompt define a personalidade, formato de resposta e idioma do agente.
# É injetado em TODAS as entradas de treinamento para garantir consistência.
SYSTEM_PROMPT = """Você é o CyberSentinel, um agente inteligente e analista SOC (Security Operations Center) sênior especialista em resposta a incidentes cibernéticos. Você analisa logs, descrições de falhas, vulnerabilidades e alertas e fornece análises estruturadas detalhadas em português do Brasil.

Ao receber um log ou incidente, estruture sua resposta RIGOROSAMENTE com as seguintes seções em Markdown:

## 🔴 Classificação da Ameaça
Identifique o tipo de ataque e defina a severidade: CRITICAL, HIGH, MEDIUM, LOW ou INFO.

## 🎯 Mapeamento MITRE ATT&CK
Indique a Tática e Técnica correspondente ao comportamento observado (ex: T1190 - Exploit Public-Facing Application).

## 🛡️ Análise de Impacto (CIA)
Descreva o impacto à Confidencialidade, Integridade e Disponibilidade.

## 📋 Plano de Resposta e Mitigação
Indique os passos práticos de contenção, erradicação e recuperação que o analista deve executar."""

print("[CyberSentinel] System Prompt definido com sucesso!")
print(f"[CyberSentinel] Tamanho do prompt: {len(SYSTEM_PROMPT.split())} palavras")

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Aplica o template de chat do Qwen3 ao tokenizador
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-instruct",
)

print("[CyberSentinel] Carregando dataset AttackQA do HuggingFace...")
dataset_raw = load_dataset("sambanovasystems/attackqa", split="train")
print(f"[CyberSentinel] Dataset carregado: {len(dataset_raw)} exemplos totais")

# ─── Divisão 70% Treino / 30% Validação ──────────────────────────────────────
print("[CyberSentinel] Dividindo dataset em 70% Treino / 30% Validação...")
dataset_split = dataset_raw.train_test_split(test_size=0.30, seed=42)

# Limita o volume para completar o treinamento dentro do Colab gratuito (~15-20 min)
LIMITE_TREINO = min(2100, len(dataset_split["train"]))
LIMITE_VAL    = min(900,  len(dataset_split["test"]))

print(f"[CyberSentinel] Treino: {LIMITE_TREINO} exemplos | Validação: {LIMITE_VAL} exemplos")

In [ ]:
def formatar_conversa(amostra):
    """
    Converte cada linha do dataset AttackQA para o formato de chat do Qwen3.
    Injeta o System Prompt do CyberSentinel em cada conversa.
    """
    pergunta = amostra["question"]
    resposta = amostra["response"]

    conversa = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": f"Analise a seguinte questão de segurança ou alerta:\n\n{pergunta}"},
        {"role": "assistant", "content": resposta},
    ]

    # Aplica o template Qwen3 e retorna o texto formatado
    texto = tokenizer.apply_chat_template(
        conversa,
        tokenize = False,
        add_generation_prompt = False,
    )
    return {"text": texto}

print("[CyberSentinel] Formatando dataset de treinamento...")
train_raw = dataset_split["train"].select(range(LIMITE_TREINO))
val_raw   = dataset_split["test"].select(range(LIMITE_VAL))

dataset_treino    = train_raw.map(formatar_conversa, batched=False)
dataset_validacao = val_raw.map(formatar_conversa, batched=False)

print(f"[CyberSentinel] Dataset formatado! Exemplo do item [0]:")
print(dataset_treino[0]['text'][:500] + "...")

## 5. 🚀 Treinamento com SFTTrainer

Configuramos o `SFTTrainer` para treinar apenas nos tokens da resposta do assistente (`train_on_responses_only`),
ignorando a loss nos tokens do usuário/sistema — técnica que melhora a qualidade do fine-tuning.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

# ─── Configuração do Treinamento ──────────────────────────────────────────────
config_treino = SFTConfig(
    dataset_text_field           = "text",
    per_device_train_batch_size  = 2,
    gradient_accumulation_steps  = 4,    # Simula batch efetivo de 8
    warmup_steps                 = 10,
    max_steps                    = 120,  # ~10-15 min na GPU T4 gratuita
                                         # Mude para num_train_epochs=1 para treino completo
    learning_rate                = 2e-4,
    logging_steps                = 10,
    eval_strategy                = "steps",
    eval_steps                   = 20,
    optim                        = "adamw_8bit",
    weight_decay                 = 0.01,
    lr_scheduler_type            = "linear",
    seed                         = 42,
    output_dir                   = "cybersentinel_checkpoints",
    report_to                    = "none",
    remove_unused_columns        = False,
    max_seq_length               = MAX_SEQ_LENGTH,
)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset_treino,
    eval_dataset  = dataset_validacao,
    args          = config_treino,
)

# Treina apenas nas respostas do assistente (ignora loss no system/user)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

print("[CyberSentinel] Trainer configurado! Iniciando treinamento...")

In [ ]:
# Mostra uso de GPU antes do treino
gpu_stats = torch.cuda.get_device_properties(0)
mem_inicio = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
mem_total  = round(gpu_stats.total_memory / 1024**3, 3)
print(f"GPU: {gpu_stats.name} | Memória Total: {mem_total} GB | Em uso: {mem_inicio} GB")

# ─── Execução do Treinamento ───────────────────────────────────────────────────
print("[CyberSentinel] ▶ Treinamento iniciado!")
metricas = trainer.train()

# Mostra uso de GPU após o treino
mem_final    = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
mem_lora     = round(mem_final - mem_inicio, 3)
tempo_min    = round(metricas.metrics['train_runtime'] / 60, 2)
print(f"\n[CyberSentinel] ✅ Treinamento concluído em {tempo_min} minutos!")
print(f"Pico de memória GPU: {mem_final} GB ({mem_lora} GB usados pelo LoRA)")

## 6. 🧪 Teste de Inferência (Validação do CyberSentinel)

Validamos o modelo fine-tuned com um log real de ataque SQL Injection.

In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)  # Otimiza os caches internos para inferência

# ─── Log de Teste Real ────────────────────────────────────────────────────────
log_teste = """
2024-01-15 03:42:17 [ALERT] POST /api/v1/users/login HTTP/1.1
Source IP: 192.168.4.82 | Status: 200 OK
Payload: username=' OR '1'='1' -- &password=anything
Database: production_customers (CRITICAL)
Affected records: potencialmente todos os registros de usuários
""".strip()

mensagens = [
    {"role": "system",  "content": SYSTEM_PROMPT},
    {"role": "user",    "content": f"Analise o seguinte log de segurança:\n\n{log_teste}"},
]

# Prepara os tokens de entrada
texto_entrada = tokenizer.apply_chat_template(
    mensagens,
    tokenize = False,
    add_generation_prompt = True,
    enable_thinking = True,   # Ativa o modo de raciocínio interno do Qwen3
)

inputs = tokenizer(texto_entrada, return_tensors="pt").to("cuda")

print("[CyberSentinel] Gerando resposta...\n")
print("=" * 70)

streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs,
    max_new_tokens    = 1024,
    temperature       = 0.7,
    top_p             = 0.8,
    top_k             = 20,
    streamer          = streamer,
)
print("\n" + "=" * 70)

## 7. 💾 Exportação para GGUF (Q4_K_M)

**Por que Q4_K_M?**
- Melhor balanço entre qualidade e tamanho de arquivo
- Compatível com execução em CPU local via `llama-cpp-python`
- O arquivo gerado terá aproximadamente **~1.1 GB**

**Após a exportação:**
1. Clique em 📁 Files (no painel esquerdo do Colab)
2. Navegue até a pasta `cybersentinel_gguf/`
3. Baixe o arquivo `cybersentinel_qwen3.Q4_K_M.gguf`
4. Coloque na pasta `backend/models/` do projeto local
5. **Renomeie para `qwen_finetuned.gguf`** (ou edite o `app.py` com o novo nome)

In [ ]:
import os

PASTA_SAIDA = "cybersentinel_gguf"
NOME_MODELO = "cybersentinel_qwen3"
QUANTIZACAO = "q4_k_m"  # Recomendado para CPU local

print(f"[CyberSentinel] Fundindo adaptadores LoRA com o modelo base...")
print(f"[CyberSentinel] Exportando para GGUF ({QUANTIZACAO.upper()})...")
print(f"[CyberSentinel] Destino: {PASTA_SAIDA}/{NOME_MODELO}.{QUANTIZACAO.upper()}.gguf")
print()

model.save_pretrained_gguf(
    PASTA_SAIDA,
    tokenizer,
    quantization_method = QUANTIZACAO,
)

# Renomeia o arquivo gerado para o nome padronizado do projeto
gguf_gerado   = os.path.join(PASTA_SAIDA, f"{PASTA_SAIDA}.{QUANTIZACAO.upper()}.gguf")
gguf_destino  = os.path.join(PASTA_SAIDA, f"{NOME_MODELO}.{QUANTIZACAO.upper()}.gguf")

if os.path.exists(gguf_gerado) and gguf_gerado != gguf_destino:
    os.rename(gguf_gerado, gguf_destino)

# Lista os arquivos na pasta de saída
print("\n[CyberSentinel] ✅ Exportação concluída! Arquivos gerados:")
for f in sorted(os.listdir(PASTA_SAIDA)):
    tamanho_mb = os.path.getsize(os.path.join(PASTA_SAIDA, f)) / (1024 * 1024)
    print(f"  📁 {f}  ({tamanho_mb:.1f} MB)")

print()
print("═" * 70)
print("📥 PRÓXIMOS PASSOS:")
print(f"  1. Baixe: {PASTA_SAIDA}/{NOME_MODELO}.{QUANTIZACAO.upper()}.gguf")
print(f"  2. Coloque em: backend/models/")
print(f"  3. Renomeie para: qwen_finetuned.gguf")
print(f"  4. Execute: start_local.bat")
print("═" * 70)

## 8. ⚡ (Opcional) Download Automático via Google Drive

Se quiser salvar diretamente no Google Drive para facilitar o download:

In [ ]:
# Cole este bloco e execute apenas se quiser salvar no Google Drive
if False:
    from google.colab import drive
    drive.mount('/content/drive')

    import shutil
    destino_drive = "/content/drive/MyDrive/CyberSentinel/"
    os.makedirs(destino_drive, exist_ok=True)

    arquivo_gguf = f"{PASTA_SAIDA}/{NOME_MODELO}.{QUANTIZACAO.upper()}.gguf"
    shutil.copy(arquivo_gguf, destino_drive)
    print(f"[CyberSentinel] ✅ Arquivo salvo no Google Drive: {destino_drive}")